In [2]:
# ============================================================
# AG News Classification using Simple RNN
# ============================================================

# -----------------------------
# Step 1: Import Libraries
# -----------------------------
import re

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# -----------------------------
# Step 2: Load Dataset
# -----------------------------
print("Loading AG News Dataset...")

dataset = load_dataset("fancyzhx/ag_news")

train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print("Training Samples :", len(train_texts))
print("Testing Samples  :", len(test_texts))


# -----------------------------
# Step 3: Clean Text
# -----------------------------
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation, numbers, special characters
    text = re.sub(r'[^a-z\s]', '', text)

    return text

print("\nCleaning text...")

train_texts = [clean_text(text) for text in train_texts]
test_texts = [clean_text(text) for text in test_texts]

print("Sample Cleaned Text:")
print(train_texts[0])


# -----------------------------
# Step 4: Tokenization
# -----------------------------
print("\nTokenizing...")

VOCAB_SIZE = 10000

tokenizer = Tokenizer(num_words=VOCAB_SIZE)

# Learn vocabulary
tokenizer.fit_on_texts(train_texts)

# Convert text to integer sequences
X_train = tokenizer.texts_to_sequences(train_texts)
X_test = tokenizer.texts_to_sequences(test_texts)

print("Sample Sequence:")
print(X_train[0])


# -----------------------------
# Step 5: Padding
# -----------------------------
print("\nPadding sequences...")

MAX_LENGTH = 50

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)


# -----------------------------
# Step 6: One-Hot Encode Labels
# -----------------------------
NUM_CLASSES = 4

y_train = to_categorical(train_labels, NUM_CLASSES)
y_test = to_categorical(test_labels, NUM_CLASSES)

print("\nSample Label:")
print(y_train[0])


# -----------------------------
# Step 7: Build Simple RNN Model
# -----------------------------
print("\nBuilding Model...")

model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=64,
        input_length=MAX_LENGTH
    )
)

# Simple RNN Layer
model.add(SimpleRNN(64))

# Output Layer
model.add(Dense(NUM_CLASSES, activation='softmax'))

model.summary()


# -----------------------------
# Step 8: Compile Model
# -----------------------------
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# -----------------------------
# Step 9: Train Model
# -----------------------------
print("\nTraining Started...\n")

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)


# -----------------------------
# Step 10: Evaluate Model
# -----------------------------
print("\nEvaluating Model...")

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\nTest Accuracy: {:.2f}%".format(accuracy * 100))


# -----------------------------
# Step 11: Predict a New News Article
# -----------------------------
label_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

sample_news = [
    "Apple launches a new AI powered iPhone with advanced technology."
]

# Clean
sample_news = [clean_text(text) for text in sample_news]

# Convert to sequence
sample_sequence = tokenizer.texts_to_sequences(sample_news)

# Pad
sample_sequence = pad_sequences(
    sample_sequence,
    maxlen=MAX_LENGTH,
    padding='post'
)

# Predict
prediction = model.predict(sample_sequence)

predicted_class = prediction.argmax()

print("\nPrediction")
print("---------------------")
print("News :", sample_news[0])
print("Category :", label_names[predicted_class])

Loading AG News Dataset...


Training Samples : 120000
Testing Samples  : 7600

Cleaning text...
Sample Cleaned Text:
wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again

Tokenizing...
Sample Sequence:
[391, 324, 1525, 99, 54, 1, 812, 23, 23, 391, 1988, 4, 34, 3893, 737, 295]

Padding sequences...
Training Shape: (120000, 50)
Testing Shape : (7600, 50)

Sample Label:
[0. 0. 1. 0.]

Building Model...


/workspaces/priyanshverma2004/wizard/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training Started...

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.8232 - loss: 0.5011 - val_accuracy: 0.8653 - val_loss: 0.3848
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8985 - loss: 0.3113 - val_accuracy: 0.8590 - val_loss: 0.4147
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9112 - loss: 0.2672 - val_accuracy: 0.8540 - val_loss: 0.4263
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9230 - loss: 0.2330 - val_accuracy: 0.8611 - val_loss: 0.4255
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9323 - loss: 0.2032 - val_accuracy: 0.8554 - val_loss: 0.4551

Evaluating Model...
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8678 - loss: 0.4088

Test Accuracy: 86.78%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step

Prediction
---------------------
News : apple launches a new ai powered iphone with advanced technology
Category : Sci/Tech
